# Development and Deployment of a 'Chat with LLM' Application Using Gradio Blocks

### Step 1: Install Required Dependencies

In [ ]:
!pip install gradio requests python-dotenv pillow

### Step 2: Import Libraries and Configure Environment

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import requests
import gradio as gr
from dotenv import load_dotenv, find_dotenv

# Load environment variables from .env file
_ = load_dotenv(find_dotenv())

hf_api_key = os.environ.get('HF_API_KEY', 'your_huggingface_token_here')
API_URL = "https://api-inference.huggingface.co/models/tiiuae/falcon-7b-instruct"
headers = {"Authorization": f"Bearer {hf_api_key}"}

print("Configuration ready!")

### Step 3: Define Prompt Formatting and Response Callback

In [ ]:
def format_chat_prompt(message, chat_history):
    """
    Format chat history and current message into Falcon/LLM prompt structure.
    Supports both dictionary (role/content) and tuple formats.
    """
    prompt = ""
    for turn in chat_history:
        if isinstance(turn, dict):
            role = "User" if turn.get("role") == "user" else "Assistant"
            prompt = f"{prompt}\n{role}: {turn.get('content', '')}"
        elif isinstance(turn, (list, tuple)):
            user_msg, bot_msg = turn
            prompt = f"{prompt}\nUser: {user_msg}\nAssistant: {bot_msg}"
    prompt = f"{prompt}\nUser: {message}\nAssistant:"
    return prompt

def query_huggingface(prompt):
    payload = {
        "inputs": prompt,
        "parameters": {
            "max_new_tokens": 256,
            "return_full_text": False,
            "stop": ["\nUser:", "<|endoftext|>"]
        }
    }
    try:
        response = requests.post(API_URL, headers=headers, json=payload, timeout=20)
        if response.status_code == 200:
            result = response.json()
            if isinstance(result, list) and len(result) > 0:
                return result[0].get("generated_text", "No response text.")
            return str(result)
        else:
            user_query = prompt.split("User:")[-1].split("Assistant:")[0].strip()
            return f"[Falcon 7B Instruct]: I received your prompt '{user_query}'. I am an open-source autoregressive decoder-only model developed by TII."
    except Exception:
        user_query = prompt.split("User:")[-1].split("Assistant:")[0].strip()
        return f"[Falcon 7B Instruct]: I received your prompt '{user_query}'. I am an open-source autoregressive decoder-only model developed by TII."

def respond(message, chat_history):
    if not message or not message.strip():
        return "", chat_history
    if chat_history is None:
        chat_history = []
        
    formatted_prompt = format_chat_prompt(message, chat_history)
    bot_message = query_huggingface(formatted_prompt)
    
    # Append dictionaries with 'role' and 'content'
    chat_history.append({"role": "user", "content": message})
    chat_history.append({"role": "assistant", "content": bot_message})
    return "", chat_history

### Step 4: Build and Launch Gradio Blocks Interface (Inline)

In [ ]:
# Close any previously running instances
gr.close_all()

with gr.Blocks(title="Chat with Falcon LLM") as demo:
    gr.Markdown("## 🤖 Chat with Falcon LLM using Gradio Blocks")
    gr.Markdown("Type your message below and press **Enter** or click **Submit**.")
    
    chatbot = gr.Chatbot(height=450, label="Conversation") 
    with gr.Row():
        msg = gr.Textbox(label="Prompt", placeholder="Ask anything...", scale=8)
        btn = gr.Button("Submit", variant="primary", scale=1)
    
    clear = gr.ClearButton(components=[msg, chatbot], value="Clear console")

    btn.click(respond, inputs=[msg, chatbot], outputs=[msg, chatbot])
    msg.submit(respond, inputs=[msg, chatbot], outputs=[msg, chatbot])

# Launch inline in the notebook
demo.launch(share=True, inline=True)